# Upload processed BM3D data to a Kaggle Dataset

Downloads the processed BM3D-denoised Harvard-GF volumes from Hugging Face, assembles
consolidated `{split}_volumes.npy` + `{split}_labels.npy`, and uploads them as a **Kaggle
Dataset** so training notebooks can just Add Data instead of re-downloading from HF.

## Kaggle setup
1. Notebook settings -> **Internet: On**.
2. Secrets: `HF_TOKEN`, `KAGGLE_USERNAME`, `KAGGLE_KEY` (username here is `trnquanghuyn`).
3. Disk: consolidated 200^3 is ~27 GB under `/kaggle/temp`; a resize step (`RES<200`) is
   available and strongly recommended to keep the uploaded dataset small/fast.

## Source
- BM3D volumes: `tqhuyen/harvard-gf-denoise-benchmark-v2`,
  `classical/bm3d/3375a321513938835d2c/volumes/{split}/shard-*.npy`.
- Labels: `tqhuyen/harvard-oct-glaucoma-200` (`{split}_labels.npy`).
- Destination: `KUP_SLUG` (default `trnquanghuyn/harvard-bm3d-200`).


In [ ]:
import os, sys, subprocess, json, time
from pathlib import Path
SMOKE = os.environ.get('KUP_SMOKE', '0') == '1'
IN_KAGGLE = Path('/kaggle').is_dir()
if not SMOKE and IN_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        _us = UserSecretsClient()
        for _k in ('HF_TOKEN', 'KAGGLE_USERNAME', 'KAGGLE_KEY'):
            _v = _us.get_secret(_k)
            if _v:
                os.environ[_k] = _v
    except Exception as _e:
        print('[secrets] kaggle_secrets unavailable:', _e)
    _need = [p for m, p in (('huggingface_hub', 'huggingface_hub'), ('kaggle', 'kaggle'))
             if __import__('importlib.util', fromlist=['find_spec']).find_spec(m) is None]
    if _need:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', *_need], check=True)
import numpy as np
print('[setup] kaggle', IN_KAGGLE, '| smoke', SMOKE)


In [ ]:
CFG = dict(
    bm3d_repo='tqhuyen/harvard-gf-denoise-benchmark-v2',
    bm3d_prefix='classical/bm3d/3375a321513938835d2c',
    labels_repo='tqhuyen/harvard-oct-glaucoma-200',
    data_dir=os.environ.get('KUP_DATA_DIR', '/kaggle/temp/bm3d_export'),
    res=int(os.environ.get('KUP_RES', '200')),
    slug=os.environ.get('KUP_SLUG', 'trnquanghuyn/harvard-bm3d-200'),
    private=os.environ.get('KUP_PRIVATE', '1') == '1',
    upload=os.environ.get('KUP_UPLOAD', '1') == '1',
    message=os.environ.get('KUP_MSG', 'BM3D-denoised Harvard-GF 200^3 consolidated'),
    mode=os.environ.get('KUP_MODE', 'all'),
    res_list=tuple(int(x) for x in os.environ.get('KUP_RES_LIST', '200,128,96').split(',')),
    methods=tuple(m for m in os.environ.get('KUP_METHODS', 'bm3d,bilateral,original').split(',') if m),
    prefix=os.environ.get('KUP_PREFIX', 'harvard'),
    owner=os.environ.get('KUP_OWNER', os.environ.get('KAGGLE_USERNAME', 'trnquanghuyn')),
    cleanup=os.environ.get('KUP_CLEANUP', '1') == '1',
)
if SMOKE:
    import tempfile
    CFG.update(data_dir=tempfile.mkdtemp(prefix='kup_'), res=4, upload=False, private=True,
               mode='all', res_list=(4, 2), methods=('bm3d', 'bilateral', 'original'), cleanup=True)
SPLITS = ('Training', 'Validation', 'Test')
Path(CFG['data_dir']).mkdir(parents=True, exist_ok=True)
print('[config]', CFG)


In [ ]:
def _hf(repo, remote, local_dir):
    from huggingface_hub import hf_hub_download
    return hf_hub_download(repo_id=repo, filename=remote, repo_type='dataset',
                           token=os.environ.get('HF_TOKEN'), local_dir=str(local_dir))

def _synthetic(root):
    root = Path(root); root.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(0)
    for split in SPLITS:
        n = 4
        v = rng.integers(0, 80, size=(n, 1, 8, 8, 8), dtype=np.uint8)
        np.save(root / f'{split}_volumes.npy', v)
        np.save(root / f'{split}_labels.npy', np.array([0, 1] * (n // 2), dtype=np.int64))

def assemble(root):
    root = Path(root)
    if all((root / f'{s}_volumes.npy').exists() and (root / f'{s}_complete.json').exists() for s in SPLITS):
        print('[data] already assembled -> reuse', root)
        return
    from huggingface_hub import HfApi
    api = HfApi(token=os.environ.get('HF_TOKEN'))
    info = api.dataset_info(CFG['bm3d_repo'], files_metadata=True)
    names = [x.rfilename for x in info.siblings]
    sizes = {x.rfilename: x.size for x in info.siblings}
    shard_dir = root / '_shards'
    shard_dir.mkdir(parents=True, exist_ok=True)
    for split in SPLITS:
        labels_path = root / f'{split}_labels.npy'
        if not labels_path.exists():
            src = _hf(CFG['labels_repo'], f'{split}_labels.npy', root / '_labels')
            np.save(labels_path, np.load(src))
        labels = np.load(labels_path)
        shards = sorted(n for n in names
                        if n.startswith(f"{CFG['bm3d_prefix']}/volumes/{split}/") and n.endswith('.npy'))
        if not shards:
            raise RuntimeError(f'no shards for {split}')
        need = sum(sizes[n] for n in shards) + 2 * 1024**3
        free = __import__('shutil').disk_usage(root).free
        if free < need:
            raise OSError(f'Need ~{need / 1024**3:.0f} GiB free, have {free / 1024**3:.1f} GiB')
        first = np.load(_hf(CFG['bm3d_repo'], shards[0], shard_dir), mmap_mode='r')
        sample = (1,) + tuple(first.shape[1:]) if first.ndim == 5 else tuple(first.shape[1:])
        out = np.lib.format.open_memmap(root / f'{split}_volumes.npy', mode='w+',
                                        dtype=np.uint8, shape=(len(labels),) + sample)
        cursor = 0
        for shard in [shards[0]] + shards[1:]:
            path = _hf(CFG['bm3d_repo'], shard, shard_dir)
            arr = np.load(path, mmap_mode='r')
            out[cursor:cursor + len(arr)] = arr
            out.flush(); cursor += len(arr)
            Path(path).unlink(missing_ok=True)
            print(f'[data] {split} {cursor}/{len(labels)}', flush=True)
        if cursor != len(labels):
            raise RuntimeError(f'{split}: {cursor} != {len(labels)}')
        with open(root / f'{split}_complete.json', 'w') as fh:
            json.dump({'complete': True, 'count': cursor, 'shape': list(out.shape)}, fh)
        del out
    print('[data] assembled under', root)

if CFG['mode'] == 'single':
    if SMOKE:
        _synthetic(Path(CFG['data_dir']))
    else:
        assemble(CFG['data_dir'])


In [ ]:
def resize_dataset(src, res):
    import torch
    import torch.nn.functional as F
    src = Path(src)
    dst = Path(str(src) + f'_r{res}')
    dst.mkdir(parents=True, exist_ok=True)
    for split in SPLITS:
        if (dst / f'{split}_complete.json').exists():
            continue
        arr = np.load(src / f'{split}_volumes.npy', mmap_mode='r')
        out = np.lib.format.open_memmap(dst / f'{split}_volumes.npy', mode='w+', dtype=np.uint8,
                                        shape=(len(arr),) + tuple(arr.shape[1:-3]) + (res, res, res))
        for i in range(len(arr)):
            item = np.asarray(arr[i])
            if item.ndim == 3:
                item = item[None]
            t = torch.from_numpy(item.astype(np.float32))[None]
            t = F.interpolate(t, (res, res, res), mode='trilinear', align_corners=False)[0]
            out[i] = t.round().clip(0, 255).to(torch.uint8).numpy()
        np.save(dst / f'{split}_labels.npy', np.load(src / f'{split}_labels.npy'))
        with open(dst / f'{split}_complete.json', 'w') as fh:
            json.dump({'complete': True, 'count': len(arr), 'shape': list(out.shape)}, fh)
        del out
        print(f'[resize] {split} -> {dst.name}', flush=True)
    return dst

EXPORT_DIR = Path(CFG['data_dir'])
if CFG['mode'] == 'single' and CFG['res'] != 200:
    EXPORT_DIR = resize_dataset(CFG['data_dir'], CFG['res'])
print('[export] dir', EXPORT_DIR)


In [ ]:
def verify(root):
    for split in SPLITS:
        v = np.load(Path(root) / f'{split}_volumes.npy', mmap_mode='r')
        y = np.load(Path(root) / f'{split}_labels.npy')
        if v.dtype != np.uint8 or len(set(v.shape[-3:])) != 1 or v.shape[0] != len(y):
            raise ValueError(f'{split}: bad {v.shape} {y.shape}')
        print(f'[verify] {split}: {v.shape} {v.dtype} pos_rate={float(y.mean()):.3f} res={v.shape[-1]}')

verify(EXPORT_DIR) if CFG['mode'] == 'single' else None


In [ ]:
def kaggle_client():
    user, key = os.environ.get('KAGGLE_USERNAME'), os.environ.get('KAGGLE_KEY')
    if not (user and key):
        raise RuntimeError('KAGGLE_USERNAME/KAGGLE_KEY secrets required')
    cfg = Path('/root/.kaggle'); cfg.mkdir(parents=True, exist_ok=True)
    jp = cfg / 'kaggle.json'
    jp.write_text(json.dumps({'username': user, 'key': key}))
    jp.chmod(0o600)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'kaggle'], check=True)
    return user

def upload(export_dir, slug, private):
    kaggle_client()
    meta = {'title': slug.split('/')[-1], 'id': slug, 'licenses': [{'name': 'CC0-1.0'}]}
    (Path(export_dir) / 'dataset-metadata.json').write_text(json.dumps(meta))
    base = ['kaggle', 'datasets']
    args = ['-p', str(export_dir), '--dir-mode', 'skip']
    create = subprocess.run(base + ['create'] + args + (['--private'] if private else []),
                            capture_output=True, text=True)
    print('[upload:create]', (create.stdout or '')[-1200:], (create.stderr or '')[-800:])
    if create.returncode != 0:
        version = subprocess.run(base + ['version'] + args + ['-m', CFG['message']],
                                 capture_output=True, text=True)
        print('[upload:version]', (version.stdout or '')[-1200:], (version.stderr or '')[-800:])
        if version.returncode != 0:
            raise RuntimeError('kaggle dataset create/version failed')
    print('[upload] done ->', 'https://www.kaggle.com/datasets/' + slug)

if CFG['mode'] == 'single':
    if SMOKE:
        print('KUP_SMOKE_OK')
    elif CFG['upload']:
        upload(EXPORT_DIR, CFG['slug'], CFG['private'])
    else:
        print('[upload] KUP_UPLOAD=0 -> skipped; artifact ready at', EXPORT_DIR)


In [ ]:
METHOD_SOURCES = {
    'bm3d': {'kind': 'shards', 'repo': CFG['bm3d_repo'], 'prefix': CFG['bm3d_prefix']},
    'bilateral': {'kind': 'files', 'repo': 'tqhuyen/harvard-oct-glaucoma-200-bilateral',
                  'suffix': '_volumes_dn', 'revision': '47632c96b206707fd6423ee5b4da159069f63eaf'},
    'original': {'kind': 'files', 'repo': 'tqhuyen/harvard-oct-glaucoma-200',
                 'suffix': '_volumes', 'revision': '939a38876b7b9313162842ef2d44b7edc2b57020'},
}

def _download(repo, filename, local_dir, revision=None):
    from huggingface_hub import hf_hub_download
    return hf_hub_download(repo_id=repo, filename=filename, repo_type='dataset',
                           revision=revision, token=os.environ.get('HF_TOKEN'), local_dir=str(local_dir))

def _mark(root, split, n, shape):
    with open(Path(root) / f'{split}_complete.json', 'w') as fh:
        json.dump({'complete': True, 'count': n, 'shape': shape}, fh)

def assemble_files(dst, repo, suffix, revision):
    dst = Path(dst); dst.mkdir(parents=True, exist_ok=True)
    for split in SPLITS:
        if (dst / f'{split}_complete.json').exists():
            continue
        src = _download(repo, f'{split}{suffix}.npy', dst, revision)
        arr = np.load(src, mmap_mode='r'); n, shape = len(arr), list(arr.shape); del arr
        os.replace(src, dst / f'{split}_volumes.npy')
        lbl = _download(repo, f'{split}_labels.npy', dst, revision)
        if Path(lbl).resolve() != (dst / f'{split}_labels.npy').resolve():
            os.replace(lbl, dst / f'{split}_labels.npy')
        _mark(dst, split, n, shape)
        print(f'[data] {repo} {split} {n}', flush=True)

def assemble_shards(dst, repo, prefix, labels_repo):
    dst = Path(dst); dst.mkdir(parents=True, exist_ok=True)
    from huggingface_hub import HfApi
    info = HfApi(token=os.environ.get('HF_TOKEN')).dataset_info(repo, files_metadata=True)
    names = [x.rfilename for x in info.siblings]
    sizes = {x.rfilename: x.size for x in info.siblings}
    for split in SPLITS:
        if (dst / f'{split}_complete.json').exists():
            continue
        shards = sorted(n for n in names if n.startswith(f'{prefix}/volumes/{split}/') and n.endswith('.npy'))
        if not shards:
            raise RuntimeError(f'no shards for {split}')
        need = sum(sizes[n] for n in shards) + 2 * 1024**3
        free = __import__('shutil').disk_usage(dst).free
        if free < need:
            raise OSError(f'Need ~{need / 1024**3:.0f} GiB free, have {free / 1024**3:.1f} GiB')
        labels_path = dst / f'{split}_labels.npy'
        if not labels_path.exists():
            lbl = _download(labels_repo, f'{split}_labels.npy', dst)
            if Path(lbl).resolve() != labels_path.resolve():
                os.replace(lbl, labels_path)
        labels = np.load(labels_path)
        first = np.load(_download(repo, shards[0], dst / '_shards'), mmap_mode='r')
        sample = (1,) + tuple(first.shape[1:]) if first.ndim == 5 else tuple(first.shape[1:])
        out = np.lib.format.open_memmap(dst / f'{split}_volumes.npy', mode='w+',
                                        dtype=np.uint8, shape=(len(labels),) + sample)
        cursor = 0
        for shard in [shards[0]] + shards[1:]:
            path = _download(repo, shard, dst / '_shards')
            arr = np.load(path, mmap_mode='r')
            out[cursor:cursor + len(arr)] = arr
            out.flush(); cursor += len(arr)
            Path(path).unlink(missing_ok=True)
            print(f'[data] {split} {cursor}/{len(labels)}', flush=True)
        if cursor != len(labels):
            raise RuntimeError(f'{split}: {cursor} != {len(labels)}')
        _mark(dst, split, cursor, list(out.shape))
        del out

def prepare_method(method):
    dst = Path(CFG['data_dir']) / method
    if all((dst / f'{s}_complete.json').exists() for s in SPLITS):
        print('[data] reuse', dst); return dst
    if SMOKE:
        _synthetic(dst); return dst
    src = METHOD_SOURCES[method]
    if src['kind'] == 'shards':
        assemble_shards(dst, src['repo'], src['prefix'], CFG['labels_repo'])
    else:
        assemble_files(dst, src['repo'], src['suffix'], src.get('revision'))
    return dst

def cube_res(root):
    return int(np.load(Path(root) / 'Training_volumes.npy', mmap_mode='r').shape[-1])


In [ ]:
if CFG['mode'] == 'all':
    for method in CFG['methods']:
        staging = prepare_method(method)
        base = cube_res(staging)
        for res in sorted(set(CFG['res_list'])):
            if res > base:
                print(f'[export] skip {method} r{res} (> source {base})'); continue
            export_dir = staging if res == base else resize_dataset(staging, res)
            verify(export_dir)
            slug = f"{CFG['owner']}/{CFG['prefix']}-{method}-{res}"
            if CFG['upload']:
                upload(export_dir, slug, CFG['private'])
            else:
                print(f'[export] ready {slug} -> {export_dir}')
            if res != base and CFG['cleanup']:
                __import__('shutil').rmtree(export_dir, ignore_errors=True)
        if CFG['cleanup'] and CFG['upload']:
            __import__('shutil').rmtree(staging, ignore_errors=True)
            print('[cleanup] removed', staging)
    print('KUP_ALL_OK')


## Notes
- `KUP_MODE=all` (default) builds **bilateral, bm3d, original** at every resolution in
  `KUP_RES_LIST` (default `200,128,96`) and uploads each as its own Kaggle Dataset:
  `<owner>/harvard-<method>-<res>` (e.g. `trnquanghuyn/harvard-bm3d-128`).
- Attach only the block you need and set `B3_KAGGLE_DATASET=<slug>` in the training notebook.
- Resume-safe (`*_complete.json`); resized copies are removed after upload to save disk.
- Uploading 200^3 for 3 variants is large/slow; set `KUP_RES_LIST=128,96` to skip 200, or
  `KUP_METHODS=bm3d` to do a single variant. `KUP_MODE=single` keeps the original single-blob flow.
- `upload` uses `--dir-mode skip` (no second full-size zip).
